# Exploring the January 2025 yellow taxi file

# Setup


In [12]:
from pyspark.sql import SparkSession, functions as F
YEAR, MONTH = 2025, 1
spark = (
    SparkSession.builder.appName("Taxi Data Transformation")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
df = spark.read.parquet("data/yellow_tripdata_2025-01.parquet")
zones = spark.read.option("header", True).option("inferSchema", True).csv("data/taxi_zone_lookup.csv")
print("loaded")
print(f"Number of rows in df: {df.count()}")
print(f"Number of rows in zones: {zones.count()}")

loaded
Number of rows in df: 3475226
Number of rows in zones: 265


## Schema

In [6]:
df.printSchema()
print("rows:", df.count(), "| columns:", len(df.columns))

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

rows: 3475226 | columns: 20


In [7]:
df.select(
    "VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "trip_distance", "PULocationID", "DOLocationID",
    "payment_type", "fare_amount", "tip_amount", "total_amount",
).show(10, truncate=False)

+--------+--------------------+---------------------+-------------+------------+------------+------------+-----------+----------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|PULocationID|DOLocationID|payment_type|fare_amount|tip_amount|total_amount|
+--------+--------------------+---------------------+-------------+------------+------------+------------+-----------+----------+------------+
|1       |2025-01-01 00:18:38 |2025-01-01 00:26:59  |1.6          |229         |237         |1           |10.0       |3.0       |18.0        |
|1       |2025-01-01 00:32:40 |2025-01-01 00:35:13  |0.5          |236         |237         |1           |5.1        |2.02      |12.12       |
|1       |2025-01-01 00:44:04 |2025-01-01 00:46:01  |0.6          |141         |141         |1           |5.1        |2.0       |12.1        |
|2       |2025-01-01 00:14:27 |2025-01-01 00:20:01  |0.52         |244         |244         |2           |7.2        |0.0       |9.7         |

## Value ranges

In [8]:
df.select("trip_distance", "fare_amount", "tip_amount", "total_amount",
          "passenger_count").summary("min", "25%", "50%", "75%", "max").show()

+-------+-------------+-----------+----------+------------+---------------+
|summary|trip_distance|fare_amount|tip_amount|total_amount|passenger_count|
+-------+-------------+-----------+----------+------------+---------------+
|    min|          0.0|     -900.0|     -86.0|      -901.0|              0|
|    25%|         0.98|        8.6|       0.0|        15.2|              1|
|    50%|         1.67|      12.12|      2.45|       19.95|              1|
|    75%|          3.1|       19.5|      3.93|       27.78|              1|
|    max|    276423.57|  863372.12|     400.0|   863380.37|              9|
+-------+-------------+-----------+----------+------------+---------------+



## Missing values

In [9]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0-----------------------
 VendorID              | 0      
 tpep_pickup_datetime  | 0      
 tpep_dropoff_datetime | 0      
 passenger_count       | 540149 
 trip_distance         | 0      
 RatecodeID            | 540149 
 store_and_fwd_flag    | 540149 
 PULocationID          | 0      
 DOLocationID          | 0      
 payment_type          | 0      
 fare_amount           | 0      
 extra                 | 0      
 mta_tax               | 0      
 tip_amount            | 0      
 tolls_amount          | 0      
 improvement_surcharge | 0      
 total_amount          | 0      
 congestion_surcharge  | 540149 
 Airport_fee           | 540149 
 cbd_congestion_fee    | 0      



## Pickup dates

In [13]:
df.agg(F.min("tpep_pickup_datetime").alias("earliest_pickup"),
       F.max("tpep_pickup_datetime").alias("latest_pickup")).show(truncate=False)

df.groupBy(F.date_format("tpep_pickup_datetime", "yyyy-MM").alias("pickup_month")) \
  .count().orderBy("pickup_month").show(50)

print(f"Expected month: {YEAR}-{MONTH:02d}")

+-------------------+-------------------+
|earliest_pickup    |latest_pickup      |
+-------------------+-------------------+
|2024-12-31 20:47:55|2025-02-01 00:00:44|
+-------------------+-------------------+

+------------+-------+
|pickup_month|  count|
+------------+-------+
|     2024-12|     21|
|     2025-01|3475204|
|     2025-02|      1|
+------------+-------+

Expected month: 2025-01


## How many rows break each rule

In [14]:
pu, do = F.col("tpep_pickup_datetime"), F.col("tpep_dropoff_datetime")
day = F.to_date(pu)
start = f"{YEAR}-{MONTH:02d}-01"
nxt = f"{YEAR + (MONTH == 12)}-{MONTH % 12 + 1:02d}-01"

df.agg(
    F.sum((pu.isNull() | do.isNull()).cast("int")).alias("null_timestamp"),
    F.sum((do < pu).cast("int")).alias("dropoff_before_pickup"),
    F.sum((F.col("fare_amount") <= 0).cast("int")).alias("fare_zero_or_negative"),
    F.sum(((day < F.lit(start).cast("date")) | (day >= F.lit(nxt).cast("date"))).cast("int")).alias("out_of_month"),
).show(vertical=True)

-RECORD 0-----------------------
 null_timestamp        | 0      
 dropoff_before_pickup | 124    
 fare_zero_or_negative | 145516 
 out_of_month          | 22     



## Examples of the bad rows

In [15]:
cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
        "fare_amount", "total_amount", "payment_type"]

print("--- fare <= 0 ---")
df.filter(F.col("fare_amount") <= 0).select(*cols).show(5, truncate=False)

print("--- dropoff before pickup ---")
df.filter(F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")).select(*cols).show(5, truncate=False)

print("--- pickup outside the month ---")
df.filter(F.date_format("tpep_pickup_datetime", "yyyy-MM") != f"{YEAR}-{MONTH:02d}").select(*cols).show(5, truncate=False)

--- fare <= 0 ---
+--------------------+---------------------+-------------+-----------+------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|total_amount|payment_type|
+--------------------+---------------------+-------------+-----------+------------+------------+
|2025-01-01 00:01:41 |2025-01-01 00:07:14  |0.71         |-7.2       |-8.54       |2           |
|2025-01-01 00:55:54 |2025-01-01 01:00:38  |0.69         |-6.5       |-11.5       |4           |
|2025-01-01 00:56:12 |2025-01-01 01:15:00  |0.97         |-16.3      |-21.3       |4           |
|2025-01-01 00:55:53 |2025-01-01 01:06:49  |1.42         |-12.1      |-17.1       |2           |
|2025-01-01 00:29:35 |2025-01-01 00:36:02  |0.6          |-7.2       |-12.2       |4           |
+--------------------+---------------------+-------------+-----------+------------+------------+
only showing top 5 rows

--- dropoff before pickup ---
+--------------------+---------------------+----------

## Duplicates

In [16]:
total, distinct = df.count(), df.distinct().count()
print(f"total rows: {total} | distinct rows: {distinct} | extra copies: {total - distinct}")

total rows: 3475226 | distinct rows: 3475226 | extra copies: 0


## Tips by payment type

In [17]:
df.groupBy("payment_type").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
).orderBy("payment_type").show()

+------------+-------+--------+-------+
|payment_type|  trips|avg_fare|avg_tip|
+------------+-------+--------+-------+
|           0| 540149|   14.59|   0.45|
|           1|2444393|   18.01|   4.11|
|           2| 390429|    16.6|    0.0|
|           3|  23773|     4.5|   0.02|
|           4|  76481|   11.43|   0.04|
|           5|      1|     0.0|    0.0|
+------------+-------+--------+-------+



## Zone lookup

In [18]:
zones.show(10, truncate=False)
print("zone rows:", zones.count())
zones.filter(F.col("LocationID").isin(264, 265)).show(truncate=False)

+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows

zone rows: 265
+----------+-------+--------------+------------+
|LocationID|Bo

## Extreme values

In [20]:
extreme_cols = ["tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount", "payment_type"]
extreme = (F.col("fare_amount") > 1000) | (F.col("trip_distance") > 1000)

print("rows with fare > 1000 or distance > 1000 miles:", df.filter(extreme).count())
df.filter(extreme).select(*extreme_cols).orderBy(F.desc("total_amount")).show(10, truncate=False)

rows with fare > 1000 or distance > 1000 miles: 119
+--------------------+-------------+-----------+------------+------------+
|tpep_pickup_datetime|trip_distance|fare_amount|total_amount|payment_type|
+--------------------+-------------+-----------+------------+------------+
|2025-01-20 12:07:18 |1.6          |863372.12  |863380.37   |4           |
|2025-01-16 12:23:14 |255.33       |2450.9     |2506.71     |2           |
|2025-01-21 05:17:57 |188.88       |1309.2     |1311.7      |2           |
|2025-01-09 12:19:06 |4020.04      |82.8       |101.05      |1           |
|2025-01-10 07:00:00 |56317.73     |60.7       |64.7        |0           |
|2025-01-31 17:14:42 |11187.2      |54.5       |61.94       |1           |
|2025-01-30 06:55:29 |44684.1      |47.5       |54.94       |1           |
|2025-01-15 09:09:00 |57707.8      |43.81      |50.99       |0           |
|2025-01-29 09:13:51 |44730.3      |44.5       |45.0        |1           |
|2025-01-08 19:11:56 |1847.61      |22.6       |